## 1. Data Overview

In [ ]:
# 1.1 Load Data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../1_data/raw/Airline_review.csv")
print("Data loaded successfully!")

In [ ]:
# 1.2 Shape & Data Types
print("=== Shape ===")
print(df.shape)

print("\n=== Data Types ===")
print(df.dtypes)

In [ ]:
# 1.3 Data Structure
df.head()

In [ ]:
# 1.4 Missing Values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_df.sort_values('missing_%', ascending=False)

> ### **Note — Skytrax Review Data Characteristics**
> 
> The table below summarizes the input fields collected from the Skytrax airline review submission form and their characteristics. **This explains the reason that there are substantial missing values in certain columns.**
> 
> For reference, see the original review form: [Skytrax Review Form](https://www.airlinequality.com/write-a-review/?type=airline)

| Column | Scale | Notes |
|---|---|---|
| Airline Name | - | 497 unique airlines |
| Review_Title | - | Supplementary for analysis |
| Review Date | - | Scraping date, not flight date |
| Verified | True/False | Whether e-ticket or boarding pass was submitted |
| Review | 150~3500 chars | **Primary column for text analysis** |
| Aircraft | - | Optional field; high missingness |
| Type Of Traveller | - | Business / Family / Couple / Solo |
| Seat Type | - | First / Business / Premium Economy / Economy |
| Route | - | Free-text input |
| Date Flown | - | Actual flight date |
| Seat Comfort | 1~5 | Required field |
| Cabin Staff Service | 1~5 | Required field |
| Food & Beverages | 1~5 + N/A | N/A = service not available |
| Ground Service | 1~5 | Required field |
| Inflight Entertainment | 1~5 + N/A | N/A = service not available |
| Wifi & Connectivity | 1~5 + N/A | N/A = service not available |
| Value For Money | 1~5 | Required field |
| Overall_Rating | 1~10 | Different scale from sub-ratings |
| Recommended | Yes/No | **Target variable** |

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# 2.1 Target Variable (Recommended)
print("=== Recommended ===")
print(df['Recommended'].value_counts())
print(df['Recommended'].value_counts(normalize=True).round(3) * 100)

In [ ]:
# 2.2 Known Categorical Columns
print("=== Verified ===")
print(df['Verified'].value_counts())

In [ ]:
print("=== Seat Type ===")
print(df['Seat Type'].value_counts(dropna=False))

In [ ]:
print("=== Type Of Traveller ===")
print(df['Type Of Traveller'].value_counts(dropna=False))

In [ ]:
# 2.3 Columns Requiring Investigation
# 2.3.1 Overall Rating
df["Overall_Rating"].value_counts(dropna=False)

> ### **Why Drop `Overall_Rating`?**
> 
> - **Missing 10**: Skytrax reviews use a 1 to 10 scale for `Overall_Rating`. The absence of 10-point ratings is unexplained and cannot be verified from the raw data alone.
> 
> - **Ambiguous 'n' values**: 842 entries contain 'n' which likely represents None or N/A, but cannot be reliably imputed or removed without introducing bias.
> 
> - **Data Leakage Risk**: `Overall_Rating` is logically correlated with the target variable `Recommended` (e.g., low ratings likely map to "no", high ratings to "yes"), which would inflate model performance and obscure the true predictive power of text-based sentiment features.

In [ ]:
# 2.3.2 Review Date (by Year)
df['Review Date'] = pd.to_datetime(df['Review Date'], format='mixed', errors='coerce')
df['Review Date'].dt.year.value_counts().sort_index()

In [ ]:
# 2.3.3 Date Flown (by Year)
df['Date Flown'] = pd.to_datetime(df['Date Flown'], format='%b-%y', errors='coerce')
df['Date Flown'].dt.year.value_counts().sort_index()

In [ ]:
# 2.3.4 Airline Name
print(f"Total unique airlines: {df['Airline Name'].nunique()}")
df['Airline Name'].value_counts().head(10)

In [ ]:
# 2.3.5 Aircraft
print(f"Total unique aircraft: {df['Aircraft'].nunique()}")
df['Aircraft'].value_counts().head(10)

In [ ]:
# 2.3.6 Route
print(f"Total unique routes: {df['Route'].nunique()}")
df['Route'].value_counts().head(10)

In [ ]:
# 2.4 Numerical Rating Columns
rating_cols = ['Seat Comfort', 'Cabin Staff Service', 'Food & Beverages',
               'Ground Service', 'Inflight Entertainment',
               'Wifi & Connectivity', 'Value For Money']

df[rating_cols].describe()

In [ ]:
# 2.5 Wifi & Connectivity - detailed check
df["Wifi & Connectivity"].value_counts(dropna=False)

## 2.6 EDA Visualizations & Insights

The following visualizations examine the target variable distribution, rating patterns, and relationships between features and the recommendation outcome. These insights directly drive the cleaning decisions in Section 3.

In [ ]:
# 2.6.1 Target Variable Distribution
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

counts = df['Recommended'].value_counts()

axes[0].bar(counts.index, counts.values, color=['#d9534f', '#5cb85c'],
            edgecolor='black', alpha=0.85)
axes[0].set_title('Recommended: Count', fontweight='bold')
axes[0].set_ylabel('Count')
for i, (label, cnt) in enumerate(counts.items()):
    axes[0].text(i, cnt + 80, str(cnt), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#d9534f', '#5cb85c'], startangle=90)
axes[1].set_title('Recommended: Proportion', fontweight='bold')

plt.suptitle('Target Variable Distribution', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

ratio = counts['no'] / counts['yes']
print(f"Class ratio  no : yes  =  {ratio:.2f} : 1")

In [ ]:
# 2.6.2 Rating Column Distributions
rating_cols = ['Seat Comfort', 'Cabin Staff Service', 'Food & Beverages',
               'Ground Service', 'Inflight Entertainment',
               'Wifi & Connectivity', 'Value For Money']

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
axes = axes.flatten()

for i, col in enumerate(rating_cols):
    ax = axes[i]
    col_data = df[col].dropna()
    ax.hist(col_data, bins=[x - 0.5 for x in range(0, 7)],
            rwidth=0.8, color='steelblue', edgecolor='black', alpha=0.8)
    ax.set_title(col, fontweight='bold', fontsize=9)
    ax.set_xlabel('Rating (0 = anomaly)')
    ax.set_xticks(range(0, 6))
    missing_pct = df[col].isna().mean() * 100
    zero_pct    = (df[col] == 0).sum() / df[col].notna().sum() * 100
    ax.text(0.98, 0.95, f'NaN: {missing_pct:.1f}%\n0-val: {zero_pct:.1f}%',
            transform=ax.transAxes, ha='right', va='top', fontsize=7.5, color='#c0392b')

axes[-1].set_visible(False)
plt.suptitle('Rating Distributions — note 0-values outside the 1–5 scale', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 2.6.3 Rating Distributions by Recommended (Box Plots)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(rating_cols):
    ax = axes[i]
    groups = [df.loc[df['Recommended'] == label, col].dropna()
              for label in ['no', 'yes']]
    bp = ax.boxplot(groups, labels=['No', 'Yes'], patch_artist=True,
                    medianprops=dict(color='red', linewidth=2))
    bp['boxes'][0].set_facecolor('#f1948a')
    bp['boxes'][1].set_facecolor('#82e0aa')
    ax.set_title(col, fontweight='bold', fontsize=9)
    ax.set_ylabel('Rating')

axes[-1].set_visible(False)
plt.suptitle('Rating Distributions: Recommended = Yes vs No', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 2.6.4 Pearson Correlation — Ratings + Recommended
rec_binary = df['Recommended'].map({'yes': 1, 'no': 0})
corr_df = df[rating_cols].copy()
corr_df['Recommended'] = rec_binary
corr = corr_df.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix: Rating Columns + Recommended', fontweight='bold')
plt.tight_layout()
plt.show()

> ### Key EDA Insights
>
> #### 1. Target Variable — Class Imbalance
> - **66.3% "no" vs 33.7% "yes"** (~2:1 ratio). This is a moderate class imbalance.
> - Downstream modelling should address this via class weighting or oversampling (e.g., SMOTE).
>
> #### 2. Zero Values in Rating Columns
> - All rating columns contain **0-values** despite the documented 1–5 scale.
> - These are out-of-range and most likely represent "N/A" responses coded differently from NaN.
> - **Action**: replace 0 with NaN before imputation.
>
> #### 3. Rating Distributions by Recommended
> - The box plots show a clear separation: **"yes" reviewers consistently rate every dimension higher** than "no" reviewers.
> - **Value For Money** and **Cabin Staff Service** have the steepest median gap — the strongest structural predictors.
>
> #### 4. Optional vs Required Services
> - **Optional services** (Food & Beverages 37%, Inflight Entertainment 53%, Wifi & Connectivity 75%): NaN = service not available. The absence itself is informative — budget/short-haul flights skew toward "no" recommendation.
> - **Required services** (Seat Comfort, Cabin Staff Service, Ground Service, Value For Money): NaN = non-response; safe to impute with the column median.
>
> #### 5. Correlation Structure
> - All ratings are positively correlated (satisfied passengers rate everything higher).
> - **Value For Money** has the highest correlation with `Recommended` (≈ 0.6–0.7), confirming it as the single most predictive rating feature.

## 3. Data Cleaning

Based on the EDA findings above, the following steps are applied in sequence:

| Step | Action | Rationale |
|------|--------|-----------|
| 3.1 | Drop columns | Remove leakage, high-cardinality, and temporally irrelevant columns |
| 3.2 | Fix 0-value anomalies | Replace out-of-range 0 ratings with NaN |
| 3.3 | Optional service ratings | Create binary availability indicators; fill NaN with 0 |
| 3.4 | Impute required ratings | Fill NaN with column median (robust to skew) |
| 3.5 | Categorical columns | Fill NaN with `'Unknown'` |
| 3.6 | Encode target | Map `Recommended`: yes → 1, no → 0 |

### 3.1 Drop Irrelevant Columns

| Column | Reason |
|--------|--------|
| `Unnamed: 0` | Duplicate row index — no predictive value |
| `Overall_Rating` | **Data leakage**: directly correlated with `Recommended` by design; 842 rows contain unexplained `'n'` values (see §2.3.1) |
| `Review_Title` | Supplementary text redundant to the full `Review` column |
| `Review Date` | Scraping date (not flight date) — not causally related to the recommendation decision |
| `Date Flown` | Flight date — temporal signal not relevant here; 16.2% missing |
| `Aircraft` | 69.2% missing; free-text with extreme cardinality — not usable as a model feature |
| `Route` | 16.5% missing; 15,000+ unique free-text values — requires geospatial engineering out of scope for this study |

In [ ]:
# 3.1 Drop irrelevant / leakage columns
drop_cols = ['Unnamed: 0', 'Overall_Rating', 'Review_Title',
             'Review Date', 'Date Flown', 'Aircraft', 'Route']

df_clean = df.drop(columns=drop_cols)

print(f"Shape  before: {df.shape}   →   after: {df_clean.shape}")
print(f"\nRetained columns ({len(df_clean.columns)}):")
for col in df_clean.columns:
    print(f"  {col}")

### 3.2 Rating Column Anomalies: Zero Values

All rating columns have `min = 0` (visible in §2.4 `describe()`), yet the Skytrax form only accepts **1–5**. These out-of-range zeros are not valid ratings — they likely represent a different coding of "N/A" or a form submission error.

**Action**: replace every `0` with `NaN` before the imputation steps, so they are treated consistently with other missing values rather than pulling the mean/median downward.

In [ ]:
# 3.2 Investigate and replace 0-values in rating columns
rating_cols = ['Seat Comfort', 'Cabin Staff Service', 'Food & Beverages',
               'Ground Service', 'Inflight Entertainment',
               'Wifi & Connectivity', 'Value For Money']

print("Zero-value counts per column (before replacement):")
for col in rating_cols:
    zeros    = (df_clean[col] == 0).sum()
    non_null = df_clean[col].notna().sum()
    pct      = zeros / non_null * 100 if non_null else 0
    print(f"  {col:<30}  {zeros:>4} zeros  ({pct:.1f}% of non-null)")

# Replace 0 with NaN — outside the valid 1-5 scale
df_clean[rating_cols] = df_clean[rating_cols].replace(0, float('nan'))
print("\nAll 0-values replaced with NaN.")

### 3.3 Optional Service Ratings: Availability Indicators

For **Food & Beverages**, **Inflight Entertainment**, and **Wifi & Connectivity**, `NaN` means the *service was not offered* — not that the passenger skipped the field. Imputing these with a statistical aggregate (mean/median) would silently assert that a service existed when it did not, distorting the model.

**Strategy**:
1. Create a binary `<service>_available` indicator column (1 = service offered, 0 = not offered).
2. Fill rating `NaN` with **0** — representing "not applicable / lowest contribution to satisfaction". This keeps the column numeric without fabricating a rating.

This turns a missing-data problem into two meaningful features the model can use.

In [ ]:
# 3.3 Create availability indicators for optional service ratings
optional_cols = {
    'Food & Beverages':       'food_beverages_available',
    'Inflight Entertainment': 'inflight_entertainment_available',
    'Wifi & Connectivity':    'wifi_connectivity_available',
}

for col, indicator in optional_cols.items():
    df_clean[indicator] = df_clean[col].notna().astype(int)
    df_clean[col]       = df_clean[col].fillna(0)
    avail_pct = df_clean[indicator].mean() * 100
    print(f"'{col}':  {indicator} — {df_clean[indicator].sum()} rows available ({avail_pct:.1f}%)")

print(f"\nShape after adding indicators: {df_clean.shape}")

### 3.4 Imputing Required Rating Columns

For **Seat Comfort**, **Cabin Staff Service**, **Ground Service**, and **Value For Money**, `NaN` reflects a non-response (the service exists but the reviewer did not rate it). These rows should be kept; imputation with the **column median** is preferred over the mean because:

- Ratings are on a discrete 1–5 scale and tend to be bimodal (very satisfied vs very unsatisfied).
- The median is robust to this skew and always resolves to a valid integer on the scale.

In [ ]:
# 3.4 Median imputation for required rating columns
required_cols = ['Seat Comfort', 'Cabin Staff Service', 'Ground Service', 'Value For Money']

for col in required_cols:
    med   = df_clean[col].median()
    n_nan = df_clean[col].isna().sum()
    df_clean[col] = df_clean[col].fillna(med)
    print(f"  '{col}': imputed {n_nan} NaN → median = {med:.1f}")

print("\nMissing values in required rating cols after imputation:")
print(df_clean[required_cols].isnull().sum())

### 3.5 Categorical Column Handling

| Column | Strategy | Rationale |
|--------|----------|-----------|
| `Seat Type` | Fill NaN → `'Unknown'` | 4.7% missing (1,096 rows); dropping would lose usable data |
| `Type Of Traveller` | Fill NaN → `'Unknown'` | 16.1% missing (3,738 rows); too large to discard |
| `Airline Name` | Retain as-is | 0% missing; 497 unique values — will be encoded in the feature engineering phase |
| `Verified` | No action needed | Already a clean boolean |

In [ ]:
# 3.5 Fill NaN in categorical columns with 'Unknown'
cat_cols = ['Seat Type', 'Type Of Traveller']

for col in cat_cols:
    n_nan = df_clean[col].isna().sum()
    df_clean[col] = df_clean[col].fillna('Unknown')
    print(f"'{col}': filled {n_nan} NaN with 'Unknown'")
    print(df_clean[col].value_counts().to_string(), "\n")

### 3.6 Encode Target Variable

`Recommended` is mapped to binary integers: **yes → 1**, **no → 0**. This is required for all standard classifiers and evaluation metrics (accuracy, ROC-AUC, F1).

In [ ]:
# 3.6 Encode target variable: yes → 1, no → 0
df_clean['Recommended'] = df_clean['Recommended'].map({'yes': 1, 'no': 0})

counts = df_clean['Recommended'].value_counts()
pcts   = df_clean['Recommended'].value_counts(normalize=True) * 100
print("Target encoding applied:")
print(counts.to_string())
print(f"\nClass balance:  0 (no) = {pcts[0]:.1f}%   1 (yes) = {pcts[1]:.1f}%")

In [ ]:
# 3.7 Final cleaned dataset overview
print("=== Final Cleaned Dataset ===")
print(f"Shape: {df_clean.shape}")

summary = pd.DataFrame({
    'dtype':     df_clean.dtypes,
    'missing':   df_clean.isnull().sum(),
    'missing_%': (df_clean.isnull().sum() / len(df_clean) * 100).round(2),
})
print("\n", summary.to_string())
print("\nSample (first 3 rows):")
df_clean.head(3)

In [ ]:
# Save cleaned dataset
import os
os.makedirs('../1_data/processed', exist_ok=True)
out_path = '../1_data/processed/airline_review_cleaned.csv'
df_clean.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print(f"Final shape: {df_clean.shape}")